# Phytclust optimized

In [ ]:
class Node:
    __slots__ = ["label", "edge_length", "parent", "children"]

    def __init__(self, label=None, edge_length=None):
        """Initialize a tree node.

        Args:
            label (str): Node label (e.g., species name or identifier).
            edge_length (float): Length of the edge connecting this node to its parent.
        """
        self.label = label  # Label for the node
        self.edge_length = edge_length  # Length of edge to parent
        self.parent = None  # Parent node
        self.children = []  # List of child nodes

    def add_child(self, child):
        """Add a child node to this node.

        Args:
            child (Node): The child node to add.
        """
        if not isinstance(child, Node):
            raise TypeError("Child must be an instance of Node.")
        self.children.append(child)
        child.parent = self

    def is_leaf(self):
        """Check if the node is a leaf (no children)."""
        return len(self.children) == 0

    def is_root(self):
        """Check if the node is the root (no parent)."""
        return self.parent is None

    def __repr__(self):
        """String representation for debugging."""
        return f"Node(label={self.label}, edge_length={self.edge_length})"

In [ ]:
import re


def parse_newick(newick):
    """Parse a Newick string into a tree structure using the Node class.

    Args:
        newick (str): A Newick format string.

    Returns:
        Node: The root of the tree.
    """
    stack = []
    current_node = None
    label_buffer = None
    tokens = re.split(
        r"([(),:;])", newick
    )  # Split Newick string into meaningful tokens

    for token in tokens:
        token = token.strip()
        if not token:
            continue

        if token == "(":
            # Start a new internal node
            new_node = Node()
            if current_node:
                current_node.add_child(new_node)
            stack.append(current_node)
            current_node = new_node
        elif token == ")":
            # Close the current node and return to its parent
            if label_buffer:
                current_node.label = label_buffer
                label_buffer = None
            current_node = stack.pop()
        elif token == ",":
            # Handle a new sibling
            if label_buffer:
                current_node.label = label_buffer
                label_buffer = None
            current_node = stack[-1]
        elif token == ":":
            # Next token is an edge length
            continue
        elif token == ";":
            # End of the entire tree description
            if label_buffer:
                current_node.label = label_buffer
                label_buffer = None
            break
        else:
            # Check if token is numeric, then it's an edge length, otherwise a label
            try:
                # Attempt to parse the token as a float (edge length)
                float(token)
                # If successful, it's an edge length
                current_node.edge_length = float(token)
            except ValueError:
                # Not a number, treat it as a label buffer
                label_buffer = token

    # Find the root
    while current_node and current_node.parent:
        current_node = current_node.parent
    return current_node

In [ ]:
#! /usr/bin/env python
from collections import deque
from copy import copy
UNSAFE_SYMBOLS = {';', '(', ')', ',', '[', ']', ':', "'"}
INORDER_NONBINARY = "Can't do inorder traversal on non-binary tree"
INVALID_NEWICK = "Tree not valid Newick tree"

class Node:
    '''``Node`` class'''
    def __init__(self, label=None, edge_length=None):
        '''``Node`` constructor

        Args:
            ``label`` (``str``): Label of this ``Node``

            ``edge_length`` (``float``): Length of the edge incident to this ``Node``

        Returns:
            ``Node`` object
        '''
        self.children = []             # list of child Node objects
        self.parent = None             # parent Node object (None for root)
        self.label = label             # label
        self.edge_length = edge_length # length of incident edge

    def __lt__(self, other):
        '''Less Than operator. Just compares labels'''
        if not isinstance(other,Node):
            raise TypeError(f"'<' not supported between instances of '{type(self).__name__}' and '{type(other).__name__}'")
        elif self.label is None and other.label is not None:
            return True
        elif other.label is None:
            return False
        try:
            return float(self.label) < float(other.label)
        except:
            return str(self.label) < str(other.label)

    def __str__(self):
        '''Represent ``Node`` as a string (currently returns ``Node`` label as a string)

        Returns:
            ``str``: string representation of this ``Node``
        '''
        if self.label is None:
            return ''
        else:
            return str(self.label)

    def __copy__(self):
        '''Copy this ``Node``

        Returns:
            ``Node``: A copy of this ``Node``
        '''
        out = Node(label=copy(self.label), edge_length=copy(self.edge_length))
        out.children = copy(self.children)
        out.parent = self.parent
        return out

    def add_child(self, child):
        '''Add child to ``Node`` object

        Args:
            ``child`` (``Node``): The child ``Node`` to be added
        '''
        if not isinstance(child, Node):
            raise TypeError("child must be a Node")
        self.children.append(child); child.parent = self

    def child_nodes(self):
        '''Return a ``list`` containing this ``Node`` object's children

        Returns:
            ``list``: A ``list`` containing this ``Node`` object's children
        '''
        return copy(self.children)

    def contract(self):
        '''Contract this ``Node`` by directly connecting its children to its parent'''
        if self.is_root():
            return
        for c in self.children:
            if self.edge_length is not None and c.edge_length is not None:
                c.edge_length += self.edge_length
            self.parent.add_child(c)
        self.parent.remove_child(self)

    def get_edge_length(self):
        '''Return the length of the edge incident to this ``Node``

        Returns:
            ``float``: The length of the edge incident to this ``Node``
        '''
        return self.edge_length

    def get_label(self):
        '''Return the label of this ``Node``

        Returns:
            ``object``: The label of this ``Node``
        '''
        return self.label

    def get_parent(self):
        '''Return the parent of this ``Node``

        Returns:
            ``Node``: The parent of this ``Node``
        '''
        return self.parent

    def is_leaf(self):
        '''Returns ``True`` if this is a leaf

        Returns:
            ``bool``: ``True`` if this is a leaf, otherwise ``False``
        '''
        return len(self.children) == 0

    def is_root(self):
        '''Returns ``True`` if this is the ``root``

        Returns:
            ``bool``: ``True`` if this is the root, otherwise ``False``
        '''
        return self.parent is None

    def newick(self):
        '''Newick string conversion starting at this ``Node`` object

        Returns:
            ``str``: Newick string conversion starting at this ``Node`` object
        '''
        for node in self.traverse_postorder():
            # handle current node's label
            if node.label is None:
                str_label = ''
            else:
                str_label = str(node.label)
                for c in UNSAFE_SYMBOLS:
                    if c in str_label:
                        str_label = f"'{str_label}'"; break

            # leaf Newick representation is just its label
            if node.is_leaf():
                node.string_rep = str_label

            # handle internal node Newick representation
            else:
                out = ['(']
                for c in node.children:
                    out.append(c.string_rep)
                    if hasattr(c, 'node_params'):
                        out.append(f'[{str(c.node_params)}]')
                    if c.edge_length is not None or hasattr(c, 'edge_params'):
                        out.append(':')
                    if hasattr(c, 'edge_params'):
                        out.append(f'[{str(c.edge_params)}]')
                    if isinstance(c.edge_length, float) and c.edge_length.is_integer():
                        out.append(str(int(c.edge_length)))
                    elif c.edge_length is not None:
                        out.append(str(c.edge_length))
                    out.append(',')
                    del c.string_rep
                out.pop() # trailing comma
                out.append(')')
                if node.label is not None:
                    out.append(str_label)
                node.string_rep = ''.join(out)
        out = self.string_rep; del self.string_rep
        return out

    def num_children(self):
        '''Returns the number of children of this ``Node``

        Returns:
            ``int``: The number of children of this ``Node``
        '''
        return len(self.children)

    def num_nodes(self, leaves=True, internal=True):
        '''Compute the total number of selected nodes in the subtree rooted by this ``Node`` (including itself)

        Args:
            ``leaves`` (``bool``): ``True`` to include leaves, otherwise ``False``

            ``internal`` (``bool``): ``True`` to include internal nodes, otherwise ``False``

        Returns:
            ``int``: The total number of selected nodes in this ``Tree``
        '''
        if not isinstance(leaves, bool):
            raise TypeError("leaves must be a bool")
        if not isinstance(internal, bool):
            raise TypeError("internal must be a bool")
        return sum((leaves and node.is_leaf()) or (internal and not node.is_leaf()) for node in self.traverse_preorder())

    def remove_child(self, child):
        '''Remove child from ``Node`` object

        Args:
            ``child`` (``Node``): The child to remove
        '''
        if not isinstance(child, Node):
            raise TypeError("child must be a Node")
        try:
            self.children.remove(child); child.parent = None
        except:
            raise RuntimeError("Attempting to remove non-existent child")

    def resolve_polytomies(self):
        '''Arbitrarily resolve polytomies below this ``Node`` with 0-lengthed edges.'''
        q = deque(); q.append(self)
        while len(q) != 0:
            node = q.popleft()
            while len(node.children) > 2:
                c1 = node.children.pop(); c2 = node.children.pop()
                nn = Node(edge_length=0); node.add_child(nn)
                nn.add_child(c1); nn.add_child(c2)
            q.extend(node.children)

    def set_edge_length(self, length):
        '''Set the length of the edge incident to this ``Node``

        Args:
            ``length``: The new length of the edge incident to this ``Node``
        '''
        try:
            self.edge_length = float(length)
        except:
            raise TypeError("length must be a float")

    def set_label(self, label):
        '''Set the label of this ``Node`` object

        Args:
            ``label``: The new label
        '''
        self.label = label

    def set_parent(self, parent):
        '''Set the parent of this ``Node`` object. Use this carefully, otherwise you may damage the structure of this ``Tree`` object.

        Args:
            ``Node``: The new parent of this ``Node``
        '''
        if not isinstance(parent, Node):
            raise TypeError("parent must be a Node")
        self.parent = parent

    def traverse_ancestors(self, include_self=True):
        '''Traverse over the ancestors of this ``Node``

        Args:
            ``include_self`` (``bool``): ``True`` to include self in the traversal, otherwise ``False``
        '''
        if not isinstance(include_self, bool):
            raise TypeError("include_self must be a bool")
        if include_self:
            c = self
        else:
            c = self.parent
        while c is not None:
            yield c; c = c.parent

    def traverse_bfs(self, include_self=True):
        '''Perform a Breadth-First Search (BFS) starting at this ``Node`` object'. Yields (``Node``, distance) tuples

        Args:
            ``include_self`` (``bool``): ``True`` to include self in the traversal, otherwise ``False``
        '''
        if not isinstance(include_self, bool):
            raise TypeError("include_self must be a bool")
        q = deque(); dist = {self: 0}; q.append((self,0))
        while len(q) != 0:
            curr = q.popleft(); yield curr
            for c in curr[0].children:
                if c not in dist:
                    if c.edge_length is None:
                        el = 0
                    else:
                        el = c.edge_length
                    dist[c] = dist[curr[0]] + el; q.append((c,dist[c]))
            if curr[0].parent is not None and curr[0].parent not in dist:
                if curr[0].edge_length is None:
                    el = 0
                else:
                    el = curr[0].edge_length
                dist[curr[0].parent] = dist[curr[0]] + el; q.append((curr[0].parent,dist[curr[0].parent]))

    def traverse_inorder(self, leaves=True, internal=True):
        '''Perform an inorder traversal starting at this ``Node`` object

        Args:
            ``leaves`` (``bool``): ``True`` to include leaves, otherwise ``False``

            ``internal`` (``bool``): ``True`` to include internal nodes, otherwise ``False``
        '''
        c = self; s = deque(); done = False
        while not done:
            if c is None:
                if len(s) == 0:
                    done = True
                else:
                    c = s.pop()
                    if (leaves and c.is_leaf()) or (internal and not c.is_leaf()):
                        yield c
                    if len(c.children) == 0:
                        c = None
                    elif len(c.children) == 2:
                        c = c.children[1]
                    else:
                        raise RuntimeError(INORDER_NONBINARY)
            else:
                s.append(c)
                if len(c.children) == 0:
                    c = None
                elif len(c.children) == 2:
                    c = c.children[0]
                else:
                    raise RuntimeError(INORDER_NONBINARY)

    def traverse_internal(self):
        '''Traverse over the internal nodes below (and including) this ``Node`` object'''
        yield from self.traverse_preorder(leaves=False)

    def traverse_leaves(self):
        '''Traverse over the leaves below this ``Node`` object'''
        yield from self.traverse_preorder(internal=False)

    def traverse_levelorder(self, leaves=True, internal=True):
        '''Perform a levelorder traversal starting at this ``Node`` object

        Args:
            ``leaves`` (``bool``): ``True`` to include leaves, otherwise ``False``

            ``internal`` (``bool``): ``True`` to include internal nodes, otherwise ``False``
        '''
        q = deque(); q.append(self)
        while len(q) != 0:
            n = q.popleft()
            if (leaves and n.is_leaf()) or (internal and not n.is_leaf()):
                yield n
            q.extend(n.children)

    def traverse_postorder(self, leaves=True, internal=True):
        '''Perform a postorder traversal starting at this ``Node`` object

        Args:
            ``leaves`` (``bool``): ``True`` to include leaves, otherwise ``False``

            ``internal`` (``bool``): ``True`` to include internal nodes, otherwise ``False``
        '''
        s1 = deque(); s2 = deque(); s1.append(self)
        while len(s1) != 0:
            n = s1.pop(); s2.append(n); s1.extend(n.children)
        while len(s2) != 0:
            n = s2.pop()
            if (leaves and n.is_leaf()) or (internal and not n.is_leaf()):
                yield n

    def traverse_preorder(self, leaves=True, internal=True):
        '''Perform a preorder traversal starting at this ``Node`` object

        Args:
            ``leaves`` (``bool``): ``True`` to include leaves, otherwise ``False``

            ``internal`` (``bool``): ``True`` to include internal nodes, otherwise ``False``
        '''
        s = deque(); s.append(self)
        while len(s) != 0:
            n = s.pop()
            if (leaves and n.is_leaf()) or (internal and not n.is_leaf()):
                yield n
            s.extend(n.children)

    def traverse_rootdistorder(self, ascending=True, leaves=True, internal=True):
        '''Perform a traversal of the ``Node`` objects in the subtree rooted at this ``Node`` in either ascending (``ascending=True``) or descending (``ascending=False``) order of distance from this ``Node``

        Args:
            ``ascending`` (``bool``): ``True`` to perform traversal in ascending distance from the root, otherwise ``False`` for descending

            ``leaves`` (``bool``): ``True`` to include leaves, otherwise ``False``

            ``internal`` (``bool``): ``True`` to include internal nodes, otherwise ``False``
        '''
        if not isinstance(ascending, bool):
            raise TypeError("ascending must be a bool")
        nodes = []; dist_from_root = {}
        for node in self.traverse_preorder():
            if node == self:
                d = 0
            else:
                d = dist_from_root[node.parent]
                if node.edge_length is not None:
                    d += node.edge_length
            dist_from_root[node] = d
            if (leaves and node.is_leaf()) or (internal and not node.is_leaf()):
                nodes.append((d,node))
        nodes.sort(reverse=(not ascending))
        yield from nodes

In [ ]:
class Tree:
    """``Tree`` class"""

    def __init__(self, is_rooted=True):
        """``Tree`` constructor"""
        if not isinstance(is_rooted, bool):
            raise TypeError("is_rooted must be a bool")
        self.root = Node()  # root Node object
        self.is_rooted = is_rooted  # boolean to see if the tree is rooted or not

In [ ]:
import re


def parse_newick(newick_str):
    """
    Parse a simplified Newick string and return the root Node of the tree.

    Args:
        newick_str (str): A Newick format string, e.g. '(A:0.1,B:0.2,(C:0.3,D:0.4):0.5);'

    Returns:
        Node: The root node of the parsed tree.
    """
    # Remove any trailing semicolon for easier parsing
    newick_str = newick_str.strip()
    if newick_str.endswith(";"):
        newick_str = newick_str[:-1]

    # We'll use a manual parse that reads characters one-by-one
    stack = []
    # Start with a root "current" Node (this might remain the root or get replaced)
    current = Node()
    root = current

    # A small buffer to accumulate label characters
    label_buffer = []

    i = 0
    length_n = len(newick_str)
    while i < length_n:
        c = newick_str[i]

        if c == "(":
            # We've encountered an internal node, so create a new child of the current node
            new_node = Node()
            current.add_child(new_node)
            # Push the current node on stack (we'll come back to it when we see ')')
            stack.append(current)
            # Move deeper into the tree
            current = new_node

            i += 1

        elif c == ")":
            # We've finished the current subtree, so if there's a leftover label, assign it
            if label_buffer:
                current.label = "".join(label_buffer).strip()
                label_buffer = []

            # Return to the parent node
            if stack:
                current = stack.pop()

            i += 1

        elif c == ",":
            # We've ended one sibling; create a new sibling under the same parent
            if label_buffer:
                current.label = "".join(label_buffer).strip()
                label_buffer = []

            # The parent of current is on top of the stack
            # or is the current if we're finishing a leaf under the same parent
            parent_node = stack[-1] if stack else current

            # Create a new sibling node
            new_node = Node()
            parent_node.add_child(new_node)
            current = new_node

            i += 1

        elif c == ":":
            # The next substring (until ',', ')' or end) should be a number => edge_length
            i += 1
            start = i
            # Read until we hit a ',', ')' or end of string
            while i < length_n and newick_str[i] not in ",)":
                i += 1
            edge_str = newick_str[start:i].strip()

            # If we had accumulated a label but didn't assign it yet, do so
            if label_buffer:
                current.label = "".join(label_buffer).strip()
                label_buffer = []

            # Assign the edge length (if it’s parseable)
            if edge_str:
                try:
                    current.edge_length = float(edge_str)
                except ValueError:
                    # If it’s not a float, could be more complex data, handle as needed
                    current.edge_length = None

        else:
            # Accumulate characters as part of the label
            label_buffer.append(c)
            i += 1

    # If there's still something in label_buffer at the very end, assign it
    if label_buffer:
        current.label = "".join(label_buffer).strip()

    # Find the true root (in case we started with a dummy and then went deeper)
    while root.parent:
        root = root.parent

    return root

if __name__ == "__main__":
    # Example Newick string
    newick_str = "(A:0.1,B:0.2,(C:0.3,D:0.4):0.5);"

    # Parse into a Node-based tree
    root = parse_newick(newick_str)

    print("Root node label:", root.label)
    print("Children of root:", [child.label for child in root.children])

    # Example: Print a simple postorder traversal of all nodes
    print("Postorder labels (all nodes):")
    for node in root.traverse_postorder(leaves=True, internal=True):
        print(f"  - label: {node.label}, edge_length: {node.edge_length}")

In [ ]:
class Node:
    __slots__ = ("name", "branch_length", "branch_support", "parent", "children")

    def __init__(self, name=None, branch_length=None, branch_support=None):
        """
        Memory-efficient tree node.

        Args:
            name (str): Name/label of the node.
            branch_length (float): Length of the edge to this node's parent.
            branch_support (float): Optional support value (e.g., bootstrap).
        """
        self.name = name
        self.branch_length = branch_length
        self.branch_support = branch_support
        self.parent = None
        self.children = []

    def add_child(self, child: "Node"):
        """Attach a child to this node, setting child's parent appropriately."""
        self.children.append(child)
        child.parent = self

    def is_leaf(self) -> bool:
        """Return True if this node has no children."""
        return len(self.children) == 0

    def __repr__(self):
        return (
            f"Node(name={self.name}, length={self.branch_length}, "
            f"support={self.branch_support})"
        )


import re

# Regex to capture the node name, optional branch length, and optional branch support
# Example matches: "A", "A:0.1", "A:0.1[90]", ":0.1[90]", etc.
NODE_PATTERN = re.compile(
    r"""
    ^
    (?P<name>[^:\[\]]*)          # node name (anything not :, [, ])
    (?: : (?P<length>[^\[\]]+) )?   # optionally :length
    (?: \[ (?P<support>[^\]]+) \] )? # optionally [support]
    $
""",
    re.VERBOSE,
)


def parse_newick(newick_str: str) -> Node:
    """
    Parse a (simplified) Newick string into a tree of Node objects.

    Supports optional branch support in the format: name:0.1[90]

    Returns:
        Node: The root node of the parsed tree.
    """
    newick_str = newick_str.strip()
    # Remove trailing semicolon if present
    if newick_str.endswith(";"):
        newick_str = newick_str[:-1]

    # Create a dummy root to start building the tree
    root = Node(name="__dummy_root__")
    stack = [root]
    current_parent = root

    label_buffer = []
    i = 0
    while i < len(newick_str):
        c = newick_str[i]

        if c == "(":
            # Start a new internal node, attach to current_parent
            new_node = Node()
            current_parent.add_child(new_node)
            stack.append(current_parent)
            current_parent = new_node
            i += 1

        elif c == ")":
            # Close current subtree, pop back to parent
            # If there's a leftover label in the buffer, parse it for this node
            if label_buffer:
                parse_node_label("".join(label_buffer).strip(), current_parent)
                label_buffer.clear()
            current_parent = stack.pop()
            i += 1

        elif c == ",":
            # Sibling: finish current label, create a sibling under the same parent
            if label_buffer:
                parse_node_label("".join(label_buffer).strip(), current_parent)
                label_buffer.clear()
            # Sibling to current_parent means go back to stack top
            parent_of_siblings = stack[-1]
            new_node = Node()
            parent_of_siblings.add_child(new_node)
            current_parent = new_node
            i += 1

        else:
            # If we see a colon or bracket or any other char, just accumulate
            # We'll parse them once we hit a delimiter like ',', ')', or end
            label_buffer.append(c)
            i += 1

    # If anything remains in label_buffer, apply it to the last node
    if label_buffer:
        parse_node_label("".join(label_buffer).strip(), current_parent)
        label_buffer.clear()

    # The actual root is the only child of our dummy root if we used a dummy
    if len(root.children) == 1:
        real_root = root.children[0]
        real_root.parent = None  # remove link to dummy
        return real_root
    else:
        # Potentially the original string had multiple top-level siblings
        # or was an empty string. We'll just return root if so.
        root.name = None  # remove dummy label
        return root


def parse_node_label(label_str: str, node: Node):
    """
    Parse a label string like 'A:0.1[90]' or 'A' or ':0.2[75]' and set
    the node's name, branch_length, branch_support accordingly.
    """
    match = NODE_PATTERN.match(label_str)
    if not match:
        # If there's no match, we can decide to skip or raise an error
        return

    name = match.group("name").strip()
    length = match.group("length")
    support = match.group("support")

    # Assign name if not empty
    if name:
        node.name = name

    # Assign edge length if parseable
    if length:
        length = length.strip()
        if length:
            try:
                node.branch_length = float(length)
            except ValueError:
                pass

    # Assign branch support if parseable
    if support:
        support = support.strip()
        if support:
            try:
                node.branch_support = float(support)
            except ValueError:
                pass


def postorder_traversal(root: Node):
    """
    Iterative postorder traversal (children before the node).
    Yields nodes in postorder sequence.
    """
    if not root:
        return
    stack1 = [root]
    stack2 = []
    while stack1:
        node = stack1.pop()
        stack2.append(node)
        # Push children onto stack1
        for child in node.children:
            stack1.append(child)
    # stack2 will now have the nodes in reverse postorder
    while stack2:
        yield stack2.pop()

In [ ]:
import io
import os
from Bio import Phylo

if __name__ == "__main__":
    # Example Newick string with optional support
    newick_str = "(((A:5, B:3)C1:6, (C:3, D:7)D1:4)A13:22, (((E:7, F:13)E12:5, G:6)B23:10, H:60)int_2:35)root:0;"

    tree = Phylo.read(io.StringIO(newick_str), "newick")
    Phylo.draw(tree)
    # Parse into a Node-based tree
    root = parse_newick(newick_str)

    print("Root node:")
    print(root)

    print("\nChildren of root:")
    for child in root.children:
        print("  -", child)

    print("\nPostorder traversal (iterative):")
    for node in postorder_traversal(root):
        print(
            f"Node name={node.name}, length={node.branch_length}, support={node.children}"
        )

In [ ]:
import random
import io
from Bio import Phylo
from Bio.Phylo.Newick import Tree, Clade

def generate_random_binary_tree(num_leaves):
    """
    Generate a random binary tree with the specified number of leaf nodes.
    """
    # Create leaf nodes with random branch lengths
    leaves = [Clade(branch_length=random.randint(1, 10), name=f"Leaf{i}") for i in range(num_leaves)]

    # Randomly combine nodes into a binary tree
    while len(leaves) > 1:
        random.shuffle(leaves)
        left = leaves.pop()
        right = leaves.pop()
        parent = Clade(branch_length=random.randint(1, 10), clades=[left, right])
        leaves.append(parent)

    # The last remaining node is the root
    root = leaves[0]
    return Tree(root=root)

def tree_to_newick(tree):
    """
    Convert a Bio.Phylo tree to a Newick string.
    """
    handle = io.StringIO()
    Phylo.write(tree, handle, "newick")
    newick_str = handle.getvalue().strip()
    handle.close()
    return newick_str


if __name__ == "__main__":
    num_leaves = 10000  # Specify the number of leaf nodes
    random_tree = generate_random_binary_tree(num_leaves)
    newick_str = tree_to_newick(random_tree)

    print("Generated Newick string:")
    print(newick_str)

    tree = Phylo.read(io.StringIO(newick_str), "newick")
    Phylo.draw(tree)

    # Parse into a Node-based tree
    root = parse_newick(newick_str)

    # print("Root node:")
    # print(root)

    # print("\nChildren of root:")
    # for child in root.children:
    #     print("  -", child)

    # print("\nPostorder traversal (iterative):")
    # for node in postorder_traversal(root):
    #     print(f"Node name={node.name}, length={node.branch_length}, support={node.children}")

In [ ]:
def postorder_traversal(root: Node):
    """
    Iterative postorder traversal (children before the node).
    Yields nodes in postorder sequence.
    """
    if not root:
        return
    stack1 = [root]
    stack2 = []
    while stack1:
        node = stack1.pop()
        stack2.append(node)
        # Push children onto stack1
        for child in node.children:
            stack1.append(child)
    # stack2 will now have the nodes in reverse postorder
    while stack2:
        yield stack2.pop()

In [ ]:
def compute_dp_postorder(root, n):
    """
    Compute a DP array for each node in the tree, storing only up to
    the size of the subtree's leaves.
    `n` is the maximum number of leaves of interest in the entire tree.

    Returns:
        dp_map: dict mapping Node -> dp_array
    """
    dp_map = {}
    back_map = {}

    # 1) Build a postorder stack using two-stack method
    stack1 = [root]
    stack2 = []
    postorder_list = []

    while stack1:
        node = stack1.pop()
        stack2.append(node)
        for child in node.children:
            stack1.append(child)
    while stack2:
        postorder_list.append(stack2.pop())

    # 2) Traverse postorder_list computing DP
    for node in postorder_list:
        if len(node.children) == 0:
            dp_map[node] = [0]
            # back_map[node] = [None]
            back_map[node] = [
                [None, None, None, None]
            ]  # Initialize backtrack info for leaf nodes
            continue

        # Otherwise, combine child DP
        if len(node.children) == 2:
            left_child, right_child = node.children
            left_dp = dp_map[left_child]
            right_dp = dp_map[right_child]

            left_size = len(left_dp)
            right_size = len(right_dp)
            total_terminals = left_size + right_size
            limit = min(n, total_terminals)

            # Prepare DP array for this node
            dp_node = [float("inf")] * limit
            # back_node = np.full((2, n), limit)
            back_node = [
                [None, None, None, None] for _ in range(limit)
            ]  # 2D array for backtrack info

            dp_node[0] = (
                left_dp[0] + right_dp[0] + cost_of_branch(node, left_size, right_size)
            )
            back_node[0] = [left_child, 0, right_child, 0]

            for i in range(1, limit):
                best_val = float('inf')
                best_split = None
                for left_i in range(i):
                    right_i = i - left_i - 1

                    # Check if valid range
                    if 0 <= left_i < left_size and 0 <= right_i < right_size:
                        # Combine DP and add any cost at the parent
                        candidate = (left_dp[left_i] + right_dp[right_i])
                        if candidate < best_val:
                            best_val = candidate
                            best_split = (left_child, left_i, right_child, right_i)

                dp_node[i] = best_val
                back_node[i] = best_split

            dp_map[node] = dp_node
            back_map[node] = back_node

            dp_map.pop(left_child, None)
            dp_map.pop(right_child, None)
            # back_map.pop(left_child, None)
            # back_map.pop(right_child, None)
        else:
            print(node.name)

    # dp_map[root] now has the final DP array for the entire tree
    return dp_map, back_map


def extract_clusters(back_map, root, k):
    """
    Extract clusters for a given k from the back_map.

    Args:
        back_map: dict mapping Node -> backtrack_array
        root: the root node of the tree
        k: the number of clusters to extract

    Returns:
        clusters: a list of clusters, where each cluster is a list of nodes
    """
    clusters = []
    stack = [(root, k)]

    while stack:
        node, remaining_clusters = stack.pop()

        if remaining_clusters == 1:
            # If only one cluster remaining, add all nodes in the subtree to the cluster
            cluster = []
            nodes_to_visit = [node]
            while nodes_to_visit:
                current_node = nodes_to_visit.pop()
                cluster.append(current_node)
                nodes_to_visit.extend(current_node.children)
            clusters.append(cluster)
        else:
            # Use the backtrack information to split the node into clusters
            backtrack_info = back_map[node][remaining_clusters - 1]
            print(backtrack_info)
            # left_i, right_i = backtrack_info
            left_child, left_i, right_child, right_i = backtrack_info

            stack.append((left_child, left_i + 1))
            stack.append((right_child, right_i + 1))

    return clusters

In [ ]:
def cost_of_branch(node, left_size, right_size):
    """
    Example cost function: node has two children with branch lengths
    node.children[0].branch_length and node.children[1].branch_length,
    and we multiply by subtree sizes.
    """
    left_bl = node.children[0].branch_length or 0
    right_bl = node.children[1].branch_length or 0
    # Summation: left subtree pays left_bl for each leaf, etc.
    return left_bl * left_size + right_bl * right_size


def count_leaves(node, leaf_count_map):
    """Recursively count leaves in the subtree rooted at node."""
    if node.is_leaf():
        leaf_count_map[node] = 1
    else:
        s = 0
        for c in node.children:
            count_leaves(c, leaf_count_map)
            s += leaf_count_map[c]
        leaf_count_map[node] = s


def reconstruct_partition(node, k, dp_map, back_map):
    """
    Recursively reconstruct how we partitioned 'k' at this node.
    Returns a string or a nested data structure describing the partition.
    """
    # If leaf or k=0, there's no further split to do
    if len(node.children) == 0 or k < 0 or k >= len(dp_map[node]):
        return f"Leaf({node.name}, k={k}, cost={dp_map[node][k] if 0<=k<len(dp_map[node]) else 'N/A'})"

    best_split = back_map[node][k]
    if best_split is None:
        # Means no valid partition, or base case
        return f"Node({node.name}, k={k}, cost={dp_map[node][k]}) [NO SPLIT]"

    # Unpack the child references
    left_child, left_i, right_child, right_i = best_split

    # Build a textual representation or nested structure
    left_str = reconstruct_partition(left_child, left_i, dp_map, back_map)
    right_str = reconstruct_partition(right_child, right_i, dp_map, back_map)

    node_cost = dp_map[node][k]
    return (
        f"Node({node.name}, k={k}, cost={node_cost}) -> "
        f"LeftSplit: [{left_str}], RightSplit: [{right_str}]"
    )


# Count leaves per subtree if not already known
leaf_count_map = {}
count_leaves(
    root, leaf_count_map
)  # -> populates leaf_count_map[node] = # of leaves under node

# Let's define 'n' = total # of leaves in the entire tree
n = leaf_count_map[root]

# Now compute DP in postorder
dp_map, back_map = compute_dp_postorder(root, n)

# dp_map[root] has the final DP array for the tree root
print("DP result for root:", dp_map[root])
print("done")

In [ ]:
def process_subtree(node, n, dp_map, assignment_map):
    children = node.children

    # 1) If leaf:
    if len(children) == 0:
        dp_map[node] = [0.0]  # Only dp_map[node][0] = 0
        assignment_map[node] = [{node: 0}]  # only i=0
        return

    # 2) Postorder: process children first
    for child in children:
        process_subtree(child, n, dp_map, assignment_map)

    # 3) Now, build dp_map[node]
    if len(children) == 2:

        left_child, right_child = children
        left_dp = dp_map[left_child]  # list of floats
        right_dp = dp_map[right_child]
        left_size = len(left_dp)
        right_size = len(right_dp)
        total_terminals = left_size + right_size
        limit = min(n, total_terminals)

        dp_node = [float("inf")] * limit
        back_node = [None] * limit  # store (left_i, right_i) or something

        # MERGE scenario at i=0
        merge_cost = left_dp[0] + right_dp[0] + cost_of_branch(node, left_size, right_size)
        dp_node[0] = merge_cost
        back_node[0] = (0, 0)  # means left_i=0, right_i=0

        # SPLIT scenario for i>0
        for i in range(1, limit):
            best_val = float("inf")
            best_pair = (0, 0)
            for left_i in range(i):
                right_i = i - left_i - 1
                if left_i < left_size and right_i < right_size and right_i >= 0:
                    candidate = left_dp[left_i] + right_dp[right_i]
                    if candidate < best_val:
                        best_val = candidate
                        best_pair = (left_i, right_i)
            dp_node[i] = min(dp_node[i], best_val)
            back_node[i] = best_pair

        # Store dp
        dp_map[node] = dp_node

        # 4) Build assignment_map[node]
        assignment_map[node] = [None] * limit

        # for each i, reconstruct how leaves are assigned
        for i in range(limit):
            (left_i, right_i) = back_node[i]  # how we combined children

            if i == 0:
                # merged => entire subtree is a single cluster => cluster 0
                # gather all leaves from children in one cluster
                assignment_map[node][i] = {}
                for leaf, _ in assignment_map[left_child][0].items():
                    assignment_map[node][i][leaf] = 0
                for leaf, _ in assignment_map[right_child][0].items():
                    assignment_map[node][i][leaf] = 0
            else:
                # split => left subtree has (left_i+1) clusters, right subtree has (right_i+1)
                # but we must offset the cluster IDs from right subtree
                left_assign = assignment_map[left_child][left_i]
                right_assign = assignment_map[right_child][right_i]

                # combine them
                merged_assign = {}
                # copy left subtree's leaf->cluster
                for leaf, c_id in left_assign.items():
                    merged_assign[leaf] = c_id
                # copy right subtree's leaf->(c_id + offset)
                offset = max(left_assign.values()) + 1 if left_assign else 0
                for leaf, c_id in right_assign.items():
                    merged_assign[leaf] = c_id + offset

                assignment_map[node][i] = merged_assign

        # 5) Pop child's data to save memory
        dp_map.pop(left_child, None)
        dp_map.pop(right_child, None)
        assignment_map.pop(left_child, None)
        assignment_map.pop(right_child, None)

    return assignment_map


dp_map = {}
assignment_map = {}
clusters = process_subtree(root, n, dp_map, assignment_map)

In [ ]:
Phyl